# 기본 베이스라인

In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

# 파일 경로 설정부
file_path = r"C:\myCode\ott-churn-prediction\kim.kwangil\preprocessing\260509_view_delete\Membership_v2.csv"

# 사용 컬럼 설정부
use_cols = [
    "price",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "payment_device",
    "is_user_verified",
    "gender",
    "age",
    "is_repurchase",
]

# 이진 변수 변환 함수부
def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()

    mapped = lowered.map(
        {
            "1": 1,
            "0": 0,
            "y": 1,
            "n": 0,
            "yes": 1,
            "no": 0,
            "true": 1,
            "false": 0,
        }
    )

    numeric = pd.to_numeric(series, errors="coerce")

    return mapped.where(mapped.notna(), numeric)

# OneHotEncoder 버전 호환 함수부
def make_onehot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False
        )

# 데이터 로드부
df = pd.read_csv(file_path, usecols=use_cols).copy()

# 숫자형 변환부
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["max_screen"] = pd.to_numeric(df["max_screen"], errors="coerce")
df["age"] = pd.to_numeric(df["age"], errors="coerce")

# 이진형 변환부
binary_cols = [
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
]

for col in binary_cols:
    df[col] = to_binary(df[col])

df["is_repurchase_num"] = to_binary(df["is_repurchase"])

# 타깃 결측 제거부
df = df[df["is_repurchase_num"].isin([0, 1])].copy()

# 입력 변수, 타깃 변수 생성부
X = df[
    [
        "price",
        "max_screen",
        "is_promotion",
        "is_churn_prevented",
        "payment_device",
        "is_user_verified",
        "gender",
        "age",
    ]
].copy()

# 양성 클래스 정의부
# is_repurchase == 0 을 예측 목표로 두기 때문에 0이면 1, 1이면 0으로 변환
y = (df["is_repurchase_num"] == 0).astype(int)

# 숫자형, 범주형 컬럼 구분부
numeric_features = [
    "price",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
    "age",
]

categorical_features = [
    "payment_device",
    "gender",
]

# 전처리 파이프라인 구성부
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_onehot_encoder()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# 학습/평가 데이터 분리부
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 모델 정의부
models = {
    "LogisticRegression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
    "GradientBoosting": GradientBoostingClassifier(
        random_state=42,
    ),
}

# 평가 수행부
results = []

for model_name, model in models.items():
    clf = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]

    result = {
        "model": model_name,
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "pr_auc": average_precision_score(y_test, y_proba),
    }

    results.append(result)

# 결과 출력부
results_df = (
    pd.DataFrame(results)
    .set_index("model")
    [["precision", "recall", "f1_score", "roc_auc", "pr_auc"]]
    .round(4)
    .sort_values("f1_score", ascending=False)
)

print("양성 클래스 기준: is_repurchase == 0")
print(results_df)

양성 클래스 기준: is_repurchase == 0
                    precision  recall  f1_score  roc_auc  pr_auc
model                                                           
LogisticRegression     0.3379  0.5768    0.4261   0.5800  0.3393
RandomForest           0.3337  0.5196    0.4064   0.5615  0.3290
GradientBoosting       0.3333  0.0008    0.0015   0.5873  0.3460


# 파생 추가

In [3]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# 파일 경로 설정부
file_paths = [
    r"C:\myCode\ott-churn-prediction\kim.kwangil\derived_variable\260510_user_features_0.csv",
    r"C:\myCode\ott-churn-prediction\kim.kwangil\derived_variable\260510_user_features_1.csv",
]

# 기존 사용 컬럼 우선순위 설정부
base_feature_candidates = [
    "price",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "payment_device",
    "is_user_verified",
    "gender",
    "age",
]

# 제외 컬럼 설정부
exclude_cols = {
    "USER_NUM",
    "USER_KEY",
    "reg_date",
    "end_date",
    "is_repurchase",
    "is_repurchase_num",
}

# 범주형 컬럼 후보 설정부
categorical_candidates = {
    "payment_device",
    "gender",
}


# 이진 변수 변환 함수부
def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()

    mapped = lowered.map(
        {
            "1": 1,
            "0": 0,
            "y": 1,
            "n": 0,
            "yes": 1,
            "no": 0,
            "true": 1,
            "false": 0,
        }
    )

    numeric = pd.to_numeric(series, errors="coerce")

    return mapped.where(mapped.notna(), numeric)


# OneHotEncoder 버전 호환 함수부
def make_onehot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )


# 데이터 로드 및 병합부
df_list = [pd.read_csv(path) for path in file_paths]
df = pd.concat(df_list, ignore_index=True).copy()

# 이진형 변환부
binary_cols = [
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
]

for col in binary_cols:
    if col in df.columns:
        df[col] = to_binary(df[col])

df["is_repurchase_num"] = to_binary(df["is_repurchase"])

# 숫자형 변환부
for col in df.columns:
    if col in categorical_candidates or col in {"USER_KEY", "reg_date", "end_date"}:
        continue

    df[col] = pd.to_numeric(df[col], errors="coerce")

# 타깃 결측 제거부
df = df[df["is_repurchase_num"].isin([0, 1])].copy()

# 입력 변수 컬럼 구성부
base_features = [
    col for col in base_feature_candidates
    if col in df.columns and col not in exclude_cols
]

extra_features = [
    col for col in df.columns
    if col not in exclude_cols and col not in base_features
]

feature_cols = base_features + extra_features

if not feature_cols:
    raise ValueError("사용 가능한 입력 변수 컬럼이 없습니다.")

# 입력 변수, 타깃 변수 생성부
X = df[feature_cols].copy()

# 양성 클래스 정의부
# is_repurchase == 0 예측 목표 설정부
y = (df["is_repurchase_num"] == 0).astype(int)

# 숫자형, 범주형 컬럼 구분부
categorical_features = [
    col for col in feature_cols
    if col in categorical_candidates and col in X.columns
]

numeric_features = [
    col for col in feature_cols
    if col not in categorical_features
]

# 전처리 파이프라인 구성부
transformers = []

if numeric_features:
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    transformers.append(("num", numeric_transformer, numeric_features))

if categorical_features:
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_onehot_encoder()),
        ]
    )
    transformers.append(("cat", categorical_transformer, categorical_features))

preprocessor = ColumnTransformer(transformers=transformers)

# 학습/평가 데이터 분리부
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 모델 정의부
models = {
    "LogisticRegression": LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=42,
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
    "GradientBoosting": GradientBoostingClassifier(
        random_state=42,
    ),
}

# 평가 수행부
results = []

for model_name, model in models.items():
    clf = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]

    result = {
        "model": model_name,
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "pr_auc": average_precision_score(y_test, y_proba),
    }

    results.append(result)

# 결과 출력부
results_df = (
    pd.DataFrame(results)
    .set_index("model")
    [["precision", "recall", "f1_score", "roc_auc", "pr_auc"]]
    .round(4)
    .sort_values("f1_score", ascending=False)
)

print("양성 클래스 기준: is_repurchase == 0")
print(results_df)


양성 클래스 기준: is_repurchase == 0
                    precision  recall  f1_score  roc_auc  pr_auc
model                                                           
GradientBoosting       0.7587  0.6694    0.7112   0.9065  0.7720
RandomForest           0.7518  0.6669    0.7068   0.8986  0.7594
LogisticRegression     0.6242  0.7984    0.7006   0.8836  0.7387


In [6]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# 파일 경로 설정부
file_paths = [
    r"C:\myCode\ott-churn-prediction\kim.kwangil\derived_variable\260510_user_features_0.csv",
    r"C:\myCode\ott-churn-prediction\kim.kwangil\derived_variable\260510_user_features_1.csv",
]

# 기존 사용 컬럼 우선순위 설정부
base_feature_candidates = [
    "price",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "payment_device",
    "is_user_verified",
    "gender",
    "age",
]

# 제외 컬럼 설정부
exclude_cols = {
    "USER_NUM",
    "USER_KEY",
    "reg_date",
    "end_date",
    "is_repurchase",
    "is_repurchase_num",
}

# 범주형 컬럼 후보 설정부
categorical_candidates = {
    "payment_device",
    "gender",
}

# VIF 기준 설정부
vif_threshold = 10.0


# 이진 변수 변환 함수부
def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()

    mapped = lowered.map(
        {
            "1": 1,
            "0": 0,
            "y": 1,
            "n": 0,
            "yes": 1,
            "no": 0,
            "true": 1,
            "false": 0,
        }
    )

    numeric = pd.to_numeric(series, errors="coerce")

    return mapped.where(mapped.notna(), numeric)


# OneHotEncoder 버전 호환 함수부
def make_onehot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )


# VIF 계산 함수부
def calculate_vif_table(df):
    rows = []

    for target_col in df.columns:
        x_cols = [col for col in df.columns if col != target_col]

        if not x_cols:
            vif_value = 1.0
        else:
            x = df[x_cols]
            y = df[target_col]

            model = LinearRegression()
            model.fit(x, y)
            r2 = model.score(x, y)

            if r2 >= 0.999999:
                vif_value = np.inf
            else:
                vif_value = 1.0 / (1.0 - r2)

        rows.append(
            {
                "feature": target_col,
                "vif": vif_value,
            }
        )

    return pd.DataFrame(rows).sort_values("vif", ascending=False).reset_index(drop=True)


# VIF 기반 제거 함수부
def remove_high_vif_features(df, threshold):
    working = df.copy()
    removed_rows = []

    while working.shape[1] > 1:
        vif_df = calculate_vif_table(working)
        max_vif = vif_df["vif"].iloc[0]

        if pd.isna(max_vif) or max_vif < threshold:
            break

        drop_feature = vif_df.iloc[0]["feature"]
        drop_vif = vif_df.iloc[0]["vif"]

        removed_rows.append(
            {
                "drop_feature": drop_feature,
                "vif": drop_vif,
            }
        )

        working = working.drop(columns=[drop_feature])

    removed_df = pd.DataFrame(removed_rows)
    final_vif_df = calculate_vif_table(working)

    return working, removed_df, final_vif_df


# 전처리기 생성 함수부
def build_preprocessor(feature_cols, categorical_candidates):
    categorical_features = [
        col for col in feature_cols
        if col in categorical_candidates
    ]

    numeric_features = [
        col for col in feature_cols
        if col not in categorical_features
    ]

    transformers = []

    if numeric_features:
        numeric_transformer = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]
        )
        transformers.append(("num", numeric_transformer, numeric_features))

    if categorical_features:
        categorical_transformer = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", make_onehot_encoder()),
            ]
        )
        transformers.append(("cat", categorical_transformer, categorical_features))

    return ColumnTransformer(transformers=transformers)


# 모델 평가 함수부
def evaluate_models(X_train, X_test, y_train, y_test, feature_cols, categorical_candidates):
    preprocessor = build_preprocessor(
        feature_cols=feature_cols,
        categorical_candidates=categorical_candidates,
    )

    models = {
        "LogisticRegression": LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=42,
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ),
        "GradientBoosting": GradientBoostingClassifier(
            random_state=42,
        ),
    }

    results = []

    for model_name, model in models.items():
        clf = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                ("model", model),
            ]
        )

        clf.fit(X_train[feature_cols], y_train)

        y_pred = clf.predict(X_test[feature_cols])
        y_proba = clf.predict_proba(X_test[feature_cols])[:, 1]

        results.append(
            {
                "model": model_name,
                "precision": precision_score(y_test, y_pred, zero_division=0),
                "recall": recall_score(y_test, y_pred, zero_division=0),
                "f1_score": f1_score(y_test, y_pred, zero_division=0),
                "roc_auc": roc_auc_score(y_test, y_proba),
                "pr_auc": average_precision_score(y_test, y_proba),
            }
        )

    return (
        pd.DataFrame(results)
        .set_index("model")
        [["precision", "recall", "f1_score", "roc_auc", "pr_auc"]]
        .round(4)
        .sort_values("f1_score", ascending=False)
    )


# 데이터 로드 및 병합부
df_list = [pd.read_csv(path) for path in file_paths]
df = pd.concat(df_list, ignore_index=True).copy()

# 이진형 변환부
binary_cols = [
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
]

for col in binary_cols:
    if col in df.columns:
        df[col] = to_binary(df[col])

df["is_repurchase_num"] = to_binary(df["is_repurchase"])

# 숫자형 변환부
for col in df.columns:
    if col in categorical_candidates or col in {"USER_KEY", "reg_date", "end_date"}:
        continue

    df[col] = pd.to_numeric(df[col], errors="coerce")

# 타깃 결측 제거부
df = df[df["is_repurchase_num"].isin([0, 1])].copy()

# 입력 변수 컬럼 구성부
base_features = [
    col for col in base_feature_candidates
    if col in df.columns and col not in exclude_cols
]

extra_features = [
    col for col in df.columns
    if col not in exclude_cols and col not in base_features
]

feature_cols = base_features + extra_features

if not feature_cols:
    raise ValueError("사용 가능한 입력 변수 컬럼이 없습니다.")

# 입력 변수, 타깃 변수 생성부
X = df[feature_cols].copy()

# 양성 클래스 정의부
# is_repurchase == 0 예측 목표 설정부
y = (df["is_repurchase_num"] == 0).astype(int)

# 학습/평가 데이터 분리부
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 숫자형, 범주형 컬럼 구분부
categorical_features = [
    col for col in feature_cols
    if col in categorical_candidates
]

numeric_features = [
    col for col in feature_cols
    if col not in categorical_features
]

# VIF 계산용 숫자형 데이터 준비부
numeric_imputer = SimpleImputer(strategy="median")
X_train_numeric = pd.DataFrame(
    numeric_imputer.fit_transform(X_train[numeric_features]),
    columns=numeric_features,
    index=X_train.index,
)

# VIF 기반 다중공선성 제거부
reduced_numeric_df, removed_vif_df, final_vif_df = remove_high_vif_features(
    X_train_numeric,
    threshold=vif_threshold,
)

final_numeric_features = reduced_numeric_df.columns.tolist()
final_feature_cols = final_numeric_features + categorical_features

# 제거 전 성능 평가부
baseline_results_df = evaluate_models(
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=feature_cols,
    categorical_candidates=categorical_candidates,
)

# 제거 후 성능 평가부
reduced_results_df = evaluate_models(
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=final_feature_cols,
    categorical_candidates=categorical_candidates,
)

# 성능 비교표 생성부
comparison_df = pd.concat(
    [
        baseline_results_df.assign(scenario="before_vif"),
        reduced_results_df.assign(scenario="after_vif"),
    ]
).reset_index()

comparison_df = comparison_df[
    ["scenario", "model", "precision", "recall", "f1_score", "roc_auc", "pr_auc"]
]

# 출력부
print("양성 클래스 기준: is_repurchase == 0")
print(f"원래 사용 컬럼 수: {len(feature_cols)}")
print(f"VIF로 제거된 숫자형 컬럼 수: {len(removed_vif_df)}")
print(f"최종 사용 컬럼 수: {len(final_feature_cols)}")
print()

print("제거 전 모델 성능")
print(baseline_results_df)
print()

print("제거 후 모델 성능")
print(reduced_results_df)
print()

print("제거 전후 성능 비교")
print(comparison_df.to_string(index=False))


양성 클래스 기준: is_repurchase == 0
원래 사용 컬럼 수: 134
VIF로 제거된 숫자형 컬럼 수: 61
최종 사용 컬럼 수: 73

제거 전 모델 성능
                    precision  recall  f1_score  roc_auc  pr_auc
model                                                           
GradientBoosting       0.7587  0.6694    0.7112   0.9065  0.7720
RandomForest           0.7518  0.6669    0.7068   0.8986  0.7594
LogisticRegression     0.6242  0.7984    0.7006   0.8836  0.7387

제거 후 모델 성능
                    precision  recall  f1_score  roc_auc  pr_auc
model                                                           
GradientBoosting       0.7671  0.6403    0.6980   0.8973  0.7576
RandomForest           0.7470  0.6500    0.6951   0.8909  0.7538
LogisticRegression     0.6130  0.7855    0.6886   0.8790  0.7272

제거 전후 성능 비교
  scenario              model  precision  recall  f1_score  roc_auc  pr_auc
before_vif   GradientBoosting     0.7587  0.6694    0.7112   0.9065  0.7720
before_vif       RandomForest     0.7518  0.6669    0.7068   0.8986  0.7594
be